# MobileNetV2 Custom CUDA Kernels — Colab (T4)

Runtime → Change runtime type → **T4 GPU**. Then run cells top to bottom.

Upload `custom-kernels.zip` (this folder) and optionally `dataset_subset.zip` when prompted.

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
!pip -q install ninja

## 1. Upload code (and optional dataset subset)

In [ ]:
from google.colab import files
up = files.upload()   # choose custom-kernels.zip (+ dataset_subset.zip)
import os
for f in up:
    if f.endswith('.zip'):
        os.system(f'unzip -q -o {f}')
%cd custom-kernels
!ls

## 2. Compile kernels + smoke test

In [ ]:
import torch
from custom_ops import load_ext
ext = load_ext(verbose=True)

x = torch.randn(2, 32, 56, 56, device='cuda').half()
w = torch.randn(32, 9, device='cuda').half()
b = torch.randn(32, device='cuda')
import torch.nn.functional as F
ref = F.relu6(F.conv2d(x.float(), w.float().view(32,1,3,3), b, stride=2, padding=1, groups=32))
for tiled in (False, True):
    y = ext.dw3x3_forward(x, w, b, 2, True, tiled)
    print('dw3x3', 'tiled' if tiled else 'naive', 'max err', (y.float()-ref).abs().max().item())
w2 = torch.randn(96, 32, device='cuda').half()
b2 = torch.randn(96, device='cuda')
ref2 = F.conv2d(x.float(), w2.float().view(96,32,1,1), b2)
for tiled in (False, True):
    y = ext.pw1x1_forward(x, w2, b2, False, tiled)
    print('pw1x1', 'tiled' if tiled else 'naive', 'max err', (y.float()-ref2).abs().max().item())

## 3. Per-layer micro-benchmark (naive vs tiled vs cuDNN)

In [ ]:
!python bench_kernels.py --batch 1 8 32 --out results_kernels.csv

## 4. End-to-end MobileNetV2 benchmark (+ accuracy if dataset_subset was uploaded)

In [ ]:
import os
data = '../dataset_subset' if os.path.isdir('../dataset_subset') else ''
!python bench_model.py --batch 1 8 32 --out results_model.csv --profile {('--data ' + data) if data else ''}

## 5. Plots

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
k = pd.read_csv('results_kernels.csv')
m = pd.read_csv('results_model.csv')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for op, a in zip(['dw3x3', 'pw1x1'], ax):
    s = k[(k.op == op) & (k.batch == 8)].set_index('desc')[['cudnn_ms', 'naive_ms', 'tiled_ms']]
    s.plot.bar(ax=a, title=f'{op} per-layer latency @ batch 8 (ms)', logy=True)
    a.tick_params(axis='x', labelsize=6)
plt.tight_layout(); plt.show()

piv = m.pivot(index='batch', columns='backend', values='img_per_s')
piv.plot.bar(figsize=(9, 4), title='MobileNetV2 throughput (img/s) on ' + torch.cuda.get_device_name(0))
plt.ylabel('img/s'); plt.tight_layout(); plt.show()
print(m.pivot(index='batch', columns='backend', values='ms_per_img').round(3))
print(m.groupby('backend')[['top1','top5']].first().round(2))

## 6. Download results

In [ ]:
from google.colab import files
files.download('results_kernels.csv'); files.download('results_model.csv')